## Creating fact view

In [0]:
%sql
CREATE OR REPLACE VIEW transport_lakehouse.gold.fact_motorcycles AS
SELECT
  tw.car_num,
  m.manufacturer_key,
  vt.vehicle_type_key,
  f.fuel_type_key,
  o.ownership_key,
  tw.model_nm,
  tw.total_weight,
  tw.prod_year,
  tw.road_entry_dt,
  1 AS vehicle_count

FROM transport_lakehouse.silver.silver_vehicles_two_wheeled tw
LEFT JOIN transport_lakehouse.gold.dim_manufacturer m
  ON CAST(tw.manufacturer_cd AS STRING) = CAST(m.manufacturer_cd AS STRING)
LEFT JOIN transport_lakehouse.gold.dim_vehicle_type vt
  ON CAST(tw.vehicle_type_cd AS STRING) = CAST(vt.vehicle_type_cd AS STRING)
     AND tw.vehicle_type_nm = vt.vehicle_type_nm
LEFT JOIN transport_lakehouse.gold.dim_fuel_type f
  ON tw.fuel_type_nm = f.fuel_type_nm
LEFT JOIN transport_lakehouse.gold.dim_ownership o
  ON tw.ownership = o.ownership


## Missing keys check

In [0]:
%sql
SELECT
  SUM(CASE WHEN manufacturer_key IS NULL THEN 1 ELSE 0 END) AS missing_manufacturer_key,
  SUM(CASE WHEN vehicle_type_key IS NULL THEN 1 ELSE 0 END) AS missing_vehicle_type_key,
  SUM(CASE WHEN fuel_type_key IS NULL THEN 1 ELSE 0 END) AS missing_fuel_type_key,
  SUM(CASE WHEN ownership_key IS NULL THEN 1 ELSE 0 END) AS missing_ownership_key
FROM transport_lakehouse.gold.fact_motorcycles
    



## Duplication check

In [0]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT car_num) AS distinct_cars
FROM transport_lakehouse.gold.fact_motorcycles

## Key null ratio check

In [0]:
%sql
SELECT
  CAST(AVG(CASE WHEN manufacturer_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS manufacturer_key_fill_rate,
  CAST(AVG(CASE WHEN ownership_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS ownership_key_fill_rate,
  CAST(AVG(CASE WHEN fuel_type_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS fuel_type_key_fill_rate,
  CAST(AVG(CASE WHEN vehicle_type_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS vehicle_type_key_fill_rate
FROM transport_lakehouse.gold.fact_motorcycles